In [7]:
import pandas as pd

train_df = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet", engine="fastparquet")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet", engine="fastparquet")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="fastparquet")

In [8]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

In [ ]:
# sort theo timestamp và bỏ đi các cột không đóng góp vào phân loại
train_df = train_df.sort_values(by=['timestamp_num'])
valid_df = valid_df.sort_values(by=['timestamp_num'])
test_df = test_df.sort_values(by=['timestamp_num'])

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import copy
import warnings
warnings.filterwarnings('ignore')

# tách label ra làm y (target) và phần còn lại là X (features)
target_col = 'label'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# số class trong tập train
n_classes = len(np.unique(y_train))

print(f"Số lượng lớp (classes) phát hiện được: {n_classes}")
print(f"Các nhãn (labels): {np.unique(y_train)}")

In [ ]:
# customtimeseriesstacking: lớp dùng để train mô hình ensemble
class CustomTimeSeriesStacking:
    def __init__(self, base_models, meta_model, n_splits=5):
        
        self.base_models = base_models
        self.meta_model = meta_model
        self.tscv = TimeSeriesSplit(n_splits=n_splits) # chia dữ liệu hoàn toàn theo thứ tự xuất hiện của các dòng
        self.n_classes = None

    # fit: hàm huấn luyện mô hình ensemble với 3 giải đoạn
    def fit(self, X_train, y_train, X_valid, y_valid, X_test):
        self.n_classes = len(np.unique(y_train))
        
        # chuyển sang numpy array
        X_tr_np = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
        y_tr_np = y_train.values if isinstance(y_train, pd.Series) else y_train
        
        X_val_np = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
        y_val_np = y_valid.values if isinstance(y_valid, pd.Series) else y_valid
        
        X_tst_np = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

        # giai đoạn 1: chia nhỏ train_df thành 5 time-series folds => học trên train+idx và đoán trên val_idx
        # Các baselearner: Mỗi baselearner sẽ được huấn luyện 5 lần (train fold 1, đoán fold 2, train fold 1+2, đoán fold 3, ..., train fold 1+2+3+4, đoán fold 5)
        # dự đoán của baselearner trên cá fold sẽ đuợc tổng hợp vào meta_X_train (dùng để huấn luyện meta-learner) => tổng hợp dự đoán của baselearner sau khi train trên tập validation và test để tạo meta_X_valid và meta_X_test (dùng để kiểm tra và dự đoán cuối cùng)
        print("--- GIAI ĐOẠN 1: TẠO OOF (OUT-OF-FOLD) ĐẶC TRƯNG BẰNG TIME-SERIES SPLIT ---")
        meta_X_train = np.zeros((len(X_tr_np), len(self.base_models) * self.n_classes))
        
        for name, model in self.base_models.items():
            print(f"> Bắt đầu sinh OOF cho {name}...")
            col_start = list(self.base_models.keys()).index(name) * self.n_classes
            col_end = col_start + self.n_classes
            
            for fold, (tr_idx, val_idx) in enumerate(self.tscv.split(X_tr_np)):
                X_f_tr, y_f_tr = X_tr_np[tr_idx], y_tr_np[tr_idx]
                X_f_val, y_f_val = X_tr_np[val_idx], y_tr_np[val_idx]
                
                fold_model = copy.deepcopy(model)
                
                # early stopping: dùng validation là phần tiếp theo của các fold đang train để làm chuẩn early stopping
                if name == 'XGBoost':
                    fold_model.set_params(early_stopping_rounds=30)
                    fold_model.fit(X_f_tr, y_f_tr, eval_set=[(X_f_val, y_f_val)], verbose=False)
                elif name == 'CatBoost':
                    fold_model.fit(X_f_tr, y_f_tr, eval_set=[(X_f_val, y_f_val)], early_stopping_rounds=30, verbose=False)
                elif name == 'LightGBM':
                    callbacks = [lgb.early_stopping(stopping_rounds=30, verbose=False)]
                    fold_model.fit(X_f_tr, y_f_tr, eval_set=[(X_f_val, y_f_val)], callbacks=callbacks)
                else: 
                    # adaBoost không hỗ trợ native early stopping trong sklearn
                    fold_model.fit(X_f_tr, y_f_tr)
                
                # meta_X_train chứa các dự đoán của base learner trên tập train trong 6 lần train trước đó
                meta_X_train[val_idx, col_start:col_end] = fold_model.predict_proba(X_f_val)
        
        # giai đoạn 2: base model được huấn luyện lại trên toàn bộ train_df với early stopping dựa trên valid_df
        print("\n--- GIAI ĐOẠN 2: HUẤN LUYỆN LẠI BASE MODELS TRÊN TOÀN BỘ TRAIN_DF (LẤY VALID_DF LÀM EARLY STOPPING) ---")
        meta_X_valid = np.zeros((len(X_valid), len(self.base_models) * self.n_classes))
        meta_X_test = np.zeros((len(X_test), len(self.base_models) * self.n_classes))
        
        for name, model in self.base_models.items():
            print(f"> Training mô hình chính cho {name}...")
            col_start = list(self.base_models.keys()).index(name) * self.n_classes
            col_end = col_start + self.n_classes
            
            if name == 'XGBoost':
                model.set_params(early_stopping_rounds=50)
                model.fit(X_tr_np, y_tr_np, eval_set=[(X_val_np, y_val_np)], verbose=False)
            elif name == 'CatBoost':
                model.fit(X_tr_np, y_tr_np, eval_set=[(X_val_np, y_val_np)], early_stopping_rounds=50, verbose=False)
            elif name == 'LightGBM':
                callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=False)]
                model.fit(X_tr_np, y_tr_np, eval_set=[(X_val_np, y_val_np)], callbacks=callbacks)
            else:
                model.fit(X_tr_np, y_tr_np)
                
            # tạo feature từ tập validation dể kiểm tra Meta-Learner, và đặc trưng cuối để test
            meta_X_valid[:, col_start:col_end] = model.predict_proba(X_val_np)
            meta_X_test[:, col_start:col_end] = model.predict_proba(X_tst_np)
        
        # giai đoạn 3: huấn luyện meta-learner trên meta_X_train (dự đoán của base learner trên tập train) với nhãn gốc y_train
        # trả về meta_X_valid và meta_X_test để đánh giá và dự đoán cuối cùng bằng meta-learner
        print("\n--- GIAI ĐOẠN 3: HUẤN LUYỆN META-LEARNER (RandomForest) ---")
        # Do TimeSeriesSplit, fold 1 train_idx sẽ KHÔNG BAO GIỜ được mang làm valid_set,
        # vì vậy các nhãn này luôn bằng 0 ở meta_X_train -> ta lọc các hàng có giá trị đó bỏ qua.
        valid_train_mask = ~np.all(meta_X_train == 0, axis=1)
        meta_X_train_filtered = meta_X_train[valid_train_mask]
        y_train_filtered = y_tr_np[valid_train_mask]
        
        print(f"Tổng mẫu Meta-Train sau khi lọc index gốc của TimeSeries: {len(meta_X_train_filtered)}")
        self.meta_model.fit(meta_X_train_filtered, y_train_filtered)
        print("\n>>> Hoàn tất quy trình huấn luyện Custom Stacking! <<<")
        
        return meta_X_valid, meta_X_test
        
    def predict(self, meta_X):
        return self.meta_model.predict(meta_X)


In [ ]:
# khởi tạo 4 Base Learners: XGBoost, CatBoost, LightGBM, AdaBoost
xgb_model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, tree_method='hist', random_state=42)
cat_model = cb.CatBoostClassifier(iterations=500, learning_rate=0.05, random_state=42, verbose=False)
lgb_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)

base_models = {
    'XGBoost': xgb_model,
    'CatBoost': cat_model,
    'LightGBM': lgb_model,
    'AdaBoost': ada_model
}

# khởi tạo Meta Learner: RandomForest
meta_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

# khởi tạo đối tượng CustomTimeSeriesStacking với 5 splits
stacking_pipeline = CustomTimeSeriesStacking(base_models=base_models, meta_model=meta_model, n_splits=5)

# bắt đầu Fit (Thay vì dùng _enc, giờ dùng trực tiếp y_train, y_valid)
meta_X_val, meta_X_test = stacking_pipeline.fit(X_train, y_train, X_valid, y_valid, X_test)

In [ ]:
# dự đoán kết quả cuối cùng trên tập validation và test bằng meta-learner
y_pred_val = stacking_pipeline.predict(meta_X_val)
y_pred_test = stacking_pipeline.predict(meta_X_test)

# báo cáo đánh giá trên tập validation và test
print("\n========== ĐÁNH GIÁ TRÊN TẬP VALIDATION ==========")
print(f"Accuracy: {accuracy_score(y_valid, y_pred_val):.4f}")
print(classification_report(y_valid, y_pred_val))

print("\n========== ĐÁNH GIÁ TRÊN TẬP TEST GỐC ==========")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(classification_report(y_test, y_pred_test))

# bản chất trong pipeline trên, meta_x_val và meata_x_test có vai trò giống hệt nhau đối với meta-learner, tuy nhiên, các baselearner đã được phép nhìn lén vào valid_df để làm chuẩn early stopping => meta_x_val sẽ cao hơn chút so với meta_x_test
# meta_x_test mới là điểm số công tâm nhất